In [82]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error

Para iniciar, se comienza con la importación de los datasets, tanto test como prueba.

In [83]:
# Cargar los datos
train_data = pd.read_csv('/home/mtumalan/Desktop/Repos/Clases/ConcentracionIA/Edoardo(Pytorch)/houseclassifier/files/train.csv', index_col='Id')
test_data = pd.read_csv('/home/mtumalan/Desktop/Repos/Clases/ConcentracionIA/Edoardo(Pytorch)/houseclassifier/files/test.csv', index_col='Id')

# Vista previa de los datos
train_data.head()


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


Una vez corroboramos que se ha importado correctamente, realizamos el preprocesamiento, en el cual eliminaremos columnas del dataset que no serán utilizadas.

Se llenan los datos numéricos vacíos con la media de la columna.

Se llenan los datos categóricos vacíos con el más repetido.

Se codifican los datos categóricos para volverlos numéricos.

In [84]:
def preprocessing(df):
    df = df.drop(['MiscFeature', 'PoolQC', 'Fence', 'Alley'], axis=1)

    # Seleccionar las columnas numéricas y categóricas
    num_df = df.select_dtypes(include='number')
    cat_df = df.select_dtypes(include='object')

    # Llenar los valores faltantes con media para las columnas numéricas
    for col in num_df.columns:
        df[col] = df[col].fillna(df[col].mean())

    # Llenar los valores faltantes con la moda para las columnas categóricas
    for col in cat_df.columns:
        df[col] = df[col].fillna(df['LotShape'].value_counts().idxmax())

    # Codificar las columnas categóricas
    for col in cat_df.columns:
        df[col] = pd.factorize(df[col])[0]

    return df

Una vez se realiza esta función para preprocesar, se preprocesan los datasets.

In [85]:
train_data = preprocessing(train_data)
test_data = preprocessing(test_data)

train_data.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,...,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,0,65.0,8450,0,0,0,0,0,0,...,0,0,0,0,0,2,2008,0,0,208500
2,20,0,80.0,9600,0,0,0,0,1,0,...,0,0,0,0,0,5,2007,0,0,181500
3,60,0,68.0,11250,0,1,0,0,0,0,...,0,0,0,0,0,9,2008,0,0,223500
4,70,0,60.0,9550,0,1,0,0,2,0,...,272,0,0,0,0,2,2006,0,1,140000
5,60,0,84.0,14260,0,1,0,0,1,0,...,0,0,0,0,0,12,2008,0,0,250000


Se crea el modelo, al cual se le agregará una capa linear inicial con activación ReLU, creando de igual forma en las capas ocultas dependiendo de la estructura dada para la misma.

Al final, para la capa de salida es una capa lineal.

In [86]:
class HousePriceModelEstimator(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(HousePriceModelEstimator, self).__init__()

        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_size, hidden_size[0]))
        self.layers.append(nn.ReLU())
        
        for i in range(1, len(hidden_size)):
            self.layers.append(nn.Linear(hidden_size[i-1], hidden_size[i]))
            self.layers.append(nn.ReLU())

        self.layers.append(nn.Linear(hidden_size[-1], 1))
        self.fc1 = nn.Sequential(*self.layers)

    def forward(self, x):
        out = self.fc1(x)
        return out

# Definir las características y el objetivo
input_size = train_data.shape[1] - 1
hidden_size = [256, 128, 64, 32]

# Crear el modelo
model = HousePriceModelEstimator(input_size, hidden_size)
print(model)

HousePriceModelEstimator(
  (layers): ModuleList(
    (0): Linear(in_features=75, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
  (fc1): Sequential(
    (0): Linear(in_features=75, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)


En esta parte se realiza el entrenamiento del modelo

In [87]:
def train_model(model, train_loader, criterion, optimizer, num_epochs):
    model.train()
    total_loss = 0
    for epoch in range(num_epochs):
        for i, (features, target) in enumerate(train_loader):
            features = features.float()
            target = target.float()

            # Forward pass
            output = model(features)
            loss = criterion(output, target)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if (epoch + 1) % 10 == 0 and i == len(train_loader) - 1:
                print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader)}')
            total_loss = 0

In [88]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_loader = DataLoader(TensorDataset(torch.tensor(train_data.iloc[:, :-1].values), torch.tensor(train_data.iloc[:, -1].values)), batch_size=32, shuffle=True)

# Entrenar el modelo
train_model(model, train_loader, criterion, optimizer, num_epochs=100)

/home/mtumalan/Desktop/Repos/Clases/ConcentracionIA/venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:538: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/home/mtumalan/Desktop/Repos/Clases/ConcentracionIA/venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:538: UserWarning: Using a target size (torch.Size([20])) that is different to the input size (torch.Size([20, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [10/100], Loss: 89824066.7826087
Epoch [20/100], Loss: 264383109.5652174
Epoch [30/100], Loss: 93152428.52173913
Epoch [40/100], Loss: 105353805.91304348
Epoch [50/100], Loss: 66400378.43478261
Epoch [60/100], Loss: 143076229.5652174
Epoch [70/100], Loss: 106569739.13043478
Epoch [80/100], Loss: 189613990.95652175
Epoch [90/100], Loss: 86958964.86956522
Epoch [100/100], Loss: 217676332.52173913


Se puede observar por la perdida de datos que no fue la mejor implementación. Ahora intentaremos con XGBoost.

In [90]:
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, max_depth=5, learning_rate=0.1)
xgb_model.fit(train_data.iloc[:, :-1], train_data.iloc[:, -1])

if 'SalePrice' in test_data.columns:
    test_data = test_data.drop(columns=['SalePrice'])

# Predecir los precios de las casas
test_data['SalePrice'] = xgb_model.predict(test_data)

# Obtener mean absolute percentage error de ambos modelos
mape_nn = mean_absolute_percentage_error(train_data.iloc[:, -1], model(torch.tensor(train_data.iloc[:, :-1].values).float()).detach().numpy())
mape_xgb = mean_absolute_percentage_error(train_data.iloc[:, -1], xgb_model.predict(train_data.iloc[:, :-1]))

print(f'MAPE NN: {mape_nn}')
print(f'MAPE XGB: {mape_xgb}')

# Guardar los resultados en un archivo CSV

test_data[['SalePrice']].to_csv('/home/mtumalan/Desktop/Repos/Clases/ConcentracionIA/Edoardo(Pytorch)/houseclassifier/files/submission.csv')

MAPE NN: 0.35295983874765086
MAPE XGB: 0.000991813196431739


Se puede observar que la implementación con XGBoost presenta un menor error de media a comparación de la red neuronal